# Movement-strategy validation

**Question.** How faithfully does each of the four movement strategies
reproduce a commanded path?

**Scope.** Standalone; the control strategy only. All values are in *controller
units* (command counts). Actuator calibration - servo ticks, spool radius, dead
zone, latency - is out of scope and belongs to `validetion/Servomotor` and
`validetion/Servo+thimble`.

**Design.** Each strategy is run exactly as the device defines it, with its own
solver - no substitutions:

| strategy | directions it can command | solver |
|---|---|---|
| `CARDINAL` | 4 | planar |
| `CARDINAL_DIAGONAL` | 8 | planar |
| `FREE_FORM` | any | planar |
| `IK` | any | 3-D mechanism |

All four follow the same commanded path (line out, one full circle, line back;
301 points). Their motor commands are decoded back into a tactor position using
the inverse of whichever solver produced them - trilateration for the planar
strategies, wire FK for IK.

**Limitation.** Commands are decoded with the solver that generated them, so
this measures strategy-induced path error, not physical mechanism accuracy.
That the wire FK genuinely inverts the controller's IK path is verified in
`tests/test_wire_forward_kinematics.py`.

Computation lives in `analysis.py` and `figures.py`; this notebook calls them,
so there is exactly one implementation.

In [1]:
import pandas as pd

from analysis import StudyConfig, comparison_table, run_study
from figures import plot_motor_commands, plot_reconstructed_paths, plot_strategy_comparison

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 50)

config = StudyConfig()
samples, metrics = run_study(config)

print(f"radius {config.radius:.0f} controller units | "
      f"{config.line_steps + config.circle_steps + config.return_steps + 1} samples per strategy")

radius 160 controller units | 301 samples per strategy


## Strategy comparison

`total_rms` is how far the tactor ended up from where it was commanded,
averaged over the path. It splits into `quantisation_rms` (what the strategy's
direction resolution costs) and `execution_rms` (what the solver plus integer
command truncation cost).

`rms_command` is not comparable between the planar strategies and IK: they
express cable deltas differently, so IK's smaller numbers reflect its solver,
not less effort.

In [2]:
comparison_table(metrics, config).round(3)

,quantisation,kinematic_model,total_rms,total_max,quantisation_rms,execution_rms,rms_command,max_step_jump,closure_error
run,,,,,,,,,
cardinal,cardinal_4,planar,60.962,120.352,61.067,0.706,100.168,223.0,2.687
cardinal_diagonal,cardinal_8,planar,32.589,72.603,32.644,0.769,100.273,121.0,2.687
free_form,none,planar,0.824,1.534,0.000,0.824,100.318,5.0,0.000
ik,none,ik,3.517,10.426,0.000,3.517,26.365,2.0,0.000


## Shape metrics, and why they mislead

Direction quantisation is radius-preserving, so every reconstructed point sits
on the commanded radius. Aspect ratio and circle-fit RMS therefore score all
four strategies as near-perfect circles while point-to-point error differs by
almost two orders of magnitude. That is why `total_rms` is the primary metric
here.

In [3]:
names = [s.value for s in config.strategies]
metrics.loc[names, ["radial_rms", "circle_fit_rms", "aspect_ratio",
                    "closure_error", "decode_invalid_steps"]].round(3)

,radial_rms,circle_fit_rms,aspect_ratio,closure_error,decode_invalid_steps
run,,,,,
cardinal,0.602,0.108,1.001,2.687,0
cardinal_diagonal,0.699,0.161,1.001,2.687,0
free_form,0.745,0.257,1.001,0.000,0
ik,3.143,1.693,0.997,0.000,0


## Figures

In [4]:
for plot in (plot_reconstructed_paths(samples, config),
             plot_strategy_comparison(metrics, config),
             plot_motor_commands(samples, config)):
    print("saved", plot.name)

saved reconstructed_paths_by_strategy.png
saved strategy_comparison.png
saved motor_commands_by_strategy.png
